# Neural scaling law analysis

In [ ]:
from __future__ import annotations

import os
import json
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from copy import deepcopy
from typing import Any, Callable
from scipy.optimize import curve_fit
from matplotlib.lines import Line2D
from matplotlib.colors import LogNorm
from matplotlib.ticker import FuncFormatter
from matplotlib.ticker import FixedLocator, ScalarFormatter

from optimetal.utils import load_plot_style
load_plot_style()

# results directory and its subdirectories (used for plots, fit parameters, and tables)
res_dir = "./results"
os.makedirs(res_dir, exist_ok=True)
fig_dir = os.path.join(res_dir, "figures")
os.makedirs(fig_dir, exist_ok=True)
table_dir = os.path.join(res_dir, "tables")
os.makedirs(table_dir, exist_ok=True)

"""
Helper functions for processing scaling law results, plotting, fitting, and building tables.
"""

def extract_processed_dirs(results: dict) -> set[str]:
    """
    Recursively walk all leaf lists in results and collect their 'study_dir'.
    """
    processed = set()
    def _walk(obj: Any) -> None:
        if isinstance(obj, dict):
            for v in obj.values():
                _walk(v)
        elif isinstance(obj, list):
            for entry in obj:
                if "study_dir" in entry:
                    processed.add(entry["study_dir"])
    _walk(results)
    return processed

def _filter_nonempty(data: dict) -> dict:
    """
    Return only keys whose value list is non-empty.
    """
    return {k: v for k, v in data.items() if v}

def _regroup_by_entry_key(data: dict, x_key: str) -> dict:
    """
    Convert a dictionary that is keyed by hyperparameters into a dictionary that is keyed by entries.
    This is useful when you want the x-axis to display a value stored in each entry. For example, you could use 'num_parameter'.
    """
    regroup = {}
    for _, entries in data.items():
        for entry in entries:
            x_val = entry[x_key]
            regroup.setdefault(x_val, []).append(entry)
    return regroup

def plot_data(
    ax: plt.Axes, 
    data: dict, 
    label: str, 
    color: str,
    key: str = "val_loss", 
    x_from_entry: str | None = None,
    error_bars = False,
    ms: int = 4,
) -> None:
    """
    Plot one scaling law curve.
    """
    data = _filter_nonempty(data)
    if x_from_entry is not None:
        data = _regroup_by_entry_key(data, x_from_entry)
    if len(data) < 2:
        raise ValueError("Need at least two points for a fit")
    sort_idx = np.argsort([int(x) for x in data.keys()])
    x = np.array(sorted([int(x) for x in data.keys()]), dtype=float)
    y_mean = np.array(
        [np.mean([entry[key] for entry in data[k]]) for k in np.array(list(data.keys()))[sort_idx]], 
        dtype=float,
    )
    if error_bars:
        y_std = np.array(
            [np.std([entry[key] for entry in data[k]]) for k in np.array(list(data.keys()))[sort_idx]], 
            dtype=float,
        )
        ax.errorbar(
            x,
            y_mean,
            yerr=y_std,
            fmt="o",
            markersize=ms,
            markeredgecolor=color,
            markerfacecolor=color,
            ecolor=color,
            capsize=4,
            linestyle="none",
            label=label,
        ) 
    else:
        ax.plot(x, y_mean, "o", markersize=ms, label=label, color=color)

def power_law(x: float, alpha: float, x0: float) -> float:
    return (x0 / x) ** alpha

def power_law_with_floor(x: float, alpha: float, x0: float, l_0: float) -> float:
    return l_0 + (x0 / x) ** alpha

def broken_power_law(x: float, alpha1: float, alpha2: float, xc: float) -> float:
    return ((xc/ x) ** (alpha2)) * ((1 + (xc / x)) ** (alpha1 - alpha2))

def broken_power_law_with_amp(x: float, alpha1: float, alpha2: float, xc: float, A: float) -> float:
    return A * ((xc/ x) ** (alpha2)) * ((1 + (xc / x)) ** (alpha1 - alpha2))

def calc_aicc(residuals: np.ndarray, k: int) -> tuple[float, float]:
    """
    https://en.wikipedia.org/wiki/Akaike_information_criterion#Comparison_with_least_squares
    """
    residuals = np.asarray(residuals, dtype=float)
    n = residuals.size
    if n <= k + 1:
        return np.nan
    rss = np.sum(residuals**2)
    sigma2_hat = rss / n
    aic = 2 * k + n * np.log(sigma2_hat)
    aicc = aic + ((2 * k ** 2 + 2 * k) / (n - k - 1))
    return aicc

def fit_scaling_law(
    data: dict, 
    func_type: str, # see below
    key: str = "best_val_loss",
    x_from_entry: str | None = None,
)-> tuple[float, float]:
    """
    Fit the specified power law function to the scaling data.
    """
    data = _filter_nonempty(data)
    if x_from_entry is not None:
        data = _regroup_by_entry_key(data, x_from_entry)
    if len(data) < 2:
        raise ValueError("Need at least two points for a fit")
    sort_idx = np.argsort([int(x) for x in data.keys()])
    x = np.array(sorted([int(x) for x in data.keys()]), dtype=float)
    y_mean = np.array(
        [np.mean([entry[key] for entry in data[k]]) for k in np.array(list(data.keys()))[sort_idx]], 
        dtype=float,
    )
    y_std = np.array(
        [np.std([entry[key] for entry in data[k]]) for k in np.array(list(data.keys()))[sort_idx]], 
        dtype=float,
    )
    # select power law function type
    if func_type == "simple":
        func = power_law
        alpha0 = 0.1 # typical starting value
        x0_0 = x[0] # characteristic scale
        p0 = [alpha0, x0_0]
        bounds = ([-np.inf, -np.inf], [np.inf, np.inf])
    elif func_type == "floor":
        func = power_law_with_floor
        alpha0 = 0.1 # typical starting value
        x0_0 = x[0] # characteristic scale
        l_00 = y_mean[-1] # smallest observed loss
        p0 = [alpha0, x0_0, l_00]
        bounds = ([-np.inf, -np.inf, 0.0], [np.inf, np.inf, np.inf])
    elif func_type == "broken":
        func = broken_power_law
        alpha0 = 0.1 # typical starting value
        xc_0 = x[0] # critical size, where scaling changes
        p0 = [alpha0, alpha0, xc_0]
        bounds = ([-np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf])
    elif func_type == "broken_with_amp":
        func = broken_power_law_with_amp
        alpha0 = 0.1 # typical starting value
        xc_0 = x[0] # critical size, where scaling changes
        A0 = 1.0 # our validation loss is around 1.0
        p0 = [alpha0, alpha0, xc_0, A0]
        bounds = ([-np.inf, -np.inf, -np.inf, -np.inf], [np.inf, np.inf, np.inf, np.inf])
    else:
        raise ValueError("Unsupported power law type")
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            popt, pcov = curve_fit(
                f=func,
                xdata=x,
                ydata=y_mean,
                p0=p0,
                sigma=y_std,
                absolute_sigma=False, # default
                bounds=bounds,
                maxfev=10000,
            )
    except RuntimeError as e:
        popt, pcov = np.full(len(p0), np.nan), np.full((len(p0), len(p0)), np.nan)
    y_fit = func(x, *popt)
    res = y_mean - y_fit
    aicc = calc_aicc(res, len(popt))
    return {
        "popt": popt, 
        "pcov": pcov, 
        "aicc": aicc,
    }
    
def get_power_law(func_type: str) -> Callable[..., float]:
    """
    Helper function to select power law functions from strings.
    """
    if func_type == "simple":
        func = power_law
    elif func_type == "floor":
        func = power_law_with_floor
    elif func_type == "broken":
        func = broken_power_law
    elif func_type == "broken_with_amp":
        func = broken_power_law_with_amp
    else:
        raise ValueError("Unsupported power law type")
    return func

def get_fit_func_str(func_type: str, X: str) -> str:
    """
    Helper function to get a latex equation of a power law functions.
    """
    if func_type == "simple":
        leg_str = rf"$L({X:s}) = \left({X:s}_0/{X:s}\right)^{{\alpha_{{{X:s}}}}}$"
    elif func_type == "floor":
        leg_str = rf"$L({X:s}) = L_\infty + \left({X:s}_0/{X:s}\right)^{{\alpha_{{{X:s}}}}}$"
    elif func_type == "broken":
        leg_str = (rf"$L({X:s}) = \left({X:s}_c/{X:s}\right)^{{\alpha_{{{X:s}, 2}}}}"
                   rf"\left(1+{X:s}_c/{X:s}\right)^{{\alpha_{{{X:s}, 1}}-\alpha_{{{X:s}, 2}}}}$")
    elif func_type == "broken_with_amp":
        leg_str = (rf"$L({X:s}) = A \cdot \left[\left({X:s}_c/{X:s}\right)^{{\alpha_{{{X:s}, 2}}}}"
                   rf"\left(1+{X:s}_c/{X:s}\right)^{{\alpha_{{{X:s}, 1}}-\alpha_{{{X:s}, 2}}}}\right]$")
    elif func_type == "kaplan":
        leg_str = (r"$L(N, D) = \left[\left(\frac{N_0}{N}\right)^{\frac{\alpha_N}{\alpha_D}} + "
                   r"\frac{D_0}{D}\right]^{\alpha_D}$")
    elif func_type == "global":
        leg_str = (r"$L(N, D) = \left[\left(\frac{N_0}{N}\right)^{\frac{\alpha_N}{\alpha_D}} + \left(\frac{D_c}{D}\right)"
                   r"\left(1 + \frac{D_c}{D}\right)^{\frac{\alpha_{D,1}}{\alpha_{D,2}} - 1}\right]^{\alpha_{D,2}}$")
    else:
        raise ValueError("Unsupported power law type")
    return leg_str

def format_val(val: float, digits: int = 2) -> str:
    """
    Helper function for string formatting, used for latex table.
    """
    return f"${val:.{digits:d}f}$"

def fmt_param(v: float, digits: int = 1, remove_trailing_zeros: bool = False) -> str:
    """
    Parameter count formatter function.
    """
    if not np.isfinite(v):
        return "---"
    v = float(v)
    if v >= 1_000_000:
        val, unit = v / 1_000_000.0, r"\,M"
        # no decimals above or equal to 100M
        if v >= 100_000_000:
            return f"${int(round(val)):d}${unit:s}"
    elif v >= 1_000:
        val, unit = v / 1_000.0, r"\,k"
    else:
        return f"${int(round(v)):d}$"
    s = f"{val:.{digits:d}f}"
    if remove_trailing_zeros:
        s = s.rstrip('0').rstrip('.')
    return f"${s:s}${unit:s}"

def build_rows_from_fits(fits: list, func_type: str, X: str = "D") -> dict:
    """ 
    Helper function for building a dictionary from the fit results of the single scaling laws.
    """
    rows = []
    for name, popt, pcov, aicc in fits:
        # default fields
        record = {
            "Model": name, 
            "Form": func_type,
            "alpha": None, 
            "alpha1": None, 
            "alpha2": None,
            f"{X:s}_0": None, 
            f"{X:s}_c": None,
            "A": None, 
            "L_0": None,
            "AICc": f"{aicc:.2f}",
        }
        # populate depending on scaling law function type
        if func_type == "simple":
            alpha, x0 = popt
            record["alpha"] = format_val(alpha)
            record[f"{X:s}_0"] = fmt_param(x0)
        elif func_type == "floor":
            alpha, x0, linf = popt
            record["alpha"] = format_val(alpha)
            record[f"{X:s}_0"] = fmt_param(x0)
            record["L_0"] = format_val(linf)
        elif func_type == "broken":
            a1, a2, xc = popt
            record["alpha1"] = format_val(a1)
            record["alpha2"] = format_val(a2)
            record[f"{X:s}_c"] = fmt_param(xc)
        elif func_type == "broken_with_amp":
            a1, a2, xc, A = popt
            record["alpha1"] = format_val(a1)
            record["alpha2"] = format_val(a2)
            record[f"{X:s}_c"] = fmt_param(xc)
            record["A"] = format_val(A)
        else:
            raise ValueError(f"Unsupported form: {func_type:s}")
        rows.append(record)
    return rows

def latex_table_from_rows(
    rows: list[dict], 
    X: str, 
    fit_func_display: bool = False,
    include_aicc: bool = False,
) -> str:
    """
    Helper function for building latex table for the single scaling law fits.
    """
    cols = [
        "Model", 
        "Form",
        "alpha", 
        "alpha1", 
        "alpha2",
        f"{X:s}_0",
        f"{X:s}_c",
        "A", 
        "L_0", 
        "AICc",
    ]
    df = pd.DataFrame(rows, columns=cols)
    display_cols = [c for c in cols if c not in ("Model", "Form")]
    df[display_cols] = df[display_cols].where(pd.notnull(df[display_cols]), "---")
    keep_cols = [c for c in df.columns if not (df[c] == "---").all()]
    df = df[keep_cols]
    col_map = {
        "Model": "Model",
        "Form": "Form",
        "alpha": rf"$\alpha_{{{X:s}}}$",
        "alpha1": rf"$\alpha_{{{X:s},1}}$",
        "alpha2": rf"$\alpha_{{{X:s},2}}$",
        f"{X:s}_0": rf"${X:s}_0$",
        f"{X:s}_c": rf"${X:s}_c$",
        "A": r"$A$",
        "L_0": r"$L_\infty$",
        "AICc": "AICc",
    }
    df = df.rename(columns=col_map)
    # optionally drop AICc column, which is not always relevant to display
    if not include_aicc:
        df = df.drop(columns=["AICc"])
    # redundant information
    func_type = rows[0]["Form"]
    if "Form" in df.columns:
        df = df.drop(columns=["Form"])
    if fit_func_display:
        # determine parameter columns
        cols_order = list(df.columns)
        if "Model" not in cols_order or "AICc" not in cols_order:
            raise ValueError("Expected 'Model' and 'AICc' columns after processing")
        param_cols = [c for c in cols_order if c not in ("Model", "AICc")]
        # build a MultiIndex for columns
        # (top level: "" over Model, "Forms" over parameters, "" over AICc)
        top = []
        bottom = []
        for c in ["Model"] + param_cols + ["AICc"]:
            if c == "Model" or c == "AICc":
                top.append("")
                bottom.append(c)
            else:
                top.append(get_fit_func_str(func_type, X))
                bottom.append(c)

        df = df[["Model"] + param_cols + ["AICc"]]
        df.columns = pd.MultiIndex.from_arrays([top, bottom])
    latex = df.to_latex(
        index=False,
        escape=False,
        longtable=False,
        multicolumn=True,
        multicolumn_format="c",
        bold_rows=False,
        column_format=len(df.columns) * r"c@{\hspace{1em}}",
    )
    if fit_func_display:
        lines = latex.splitlines()
        for i, line in enumerate(lines):
            if line.strip().startswith("Model"):
                lines.insert(i, rf"\cmidrule(lr){{2-{len(param_cols) + 1}}}")
                break
        latex = "\n".join(lines)
    return latex

def grid_to_vectors(
    grid_dict: dict,
    metric_key: str = "val_loss",
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Helper function that converts data for the scaling law L(D,N) into a more manageable format.
    """
    data_keys = sorted([int(x) for x in grid_dict.keys()])
    param_keys = sorted([int(x) for x in grid_dict[str(data_keys[0])].keys()])
    datapoints, widths, num_parameters, mus, stds = [], [], [], [], []
    num_d = len(data_keys)
    num_n = len(param_keys)
    for dk in data_keys:
        by_n = grid_dict[str(dk)]
        for wk in param_keys:
            entries = by_n[str(wk)]
            vals = np.array([e[metric_key] for e in entries])
            datapoints.append(dk)
            widths.append(wk)
            num_parameters.append(entries[0]["num_parameter"])
            mus.append(np.mean(vals))
            stds.append(np.std(vals))
    datapoints = np.array(datapoints).reshape(num_d, num_n)
    widths = np.array(widths).reshape(num_d, num_n)
    num_parameters = np.array(num_parameters).reshape(num_d, num_n)
    mus = np.array(mus).reshape(num_d, num_n)
    stds = np.array(stds).reshape(num_d, num_n)
    return datapoints, widths, num_parameters, mus, stds

def global_scaling_fit_kaplan(X, alpha_n, alpha_d1, alpha_d2, Nc, Dc):
    """
    Combination of broken data scaling and simple power law parameter scaling following the form of Kaplan et al.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        D, N = X
        return ((Nc / N) ** (alpha_n / alpha_d2) + (Dc / D) * (1 + Dc / D) ** (alpha_d1/alpha_d2 - 1)) ** alpha_d2
    
def global_scaling_fit_hoffmann(X, alpha_n, alpha_d1, alpha_d2, Nc, Dc, l_0):
    """
    Combination of broken data scaling and simple power law parameter scaling following the form of Hoffmann et al.
    """
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        D, N = X
        return l_0 + (Nc / N) ** (alpha_n) + ((Dc / D) ** (alpha_d2)) * ((1 + Dc / D) ** (alpha_d1 - alpha_d2))

# Load 1D NSL data

In [ ]:
"""
Load the data for the invariant models.
"""

# path of the JSON file where results are stored
json_path = os.path.join("research", "scaling_data", "scaling_results.json")

# load the results if the file exists, otherwise print a warning
if os.path.exists(json_path): 
    with open(json_path, "r") as f:
        results = json.load(f)
    print(f"Loaded results for {len(extract_processed_dirs(results)):d} models from JSON file")
else:
    raise FileNotFoundError(f"File not found: {json_path:s}.")

"""
Load the data for the equivariant model.
"""

# path of the JSON file where results are stored
json_path_e3 = os.path.join("research_e3", "scaling_data", "scaling_results.json")

# load the results if the file exists, otherwise print a warning
if os.path.exists(json_path_e3): 
    with open(json_path_e3, "r") as f: 
        results_e3 = json.load(f)
    print(f"Loaded results for {len(extract_processed_dirs(results_e3)):d} models from JSON file")
else:
    raise FileNotFoundError(f"File not found: {json_path_e3:s}.")

# process the data for the equivariant model to be in the same format as the invariant models
for lmax in results_e3.keys():
    # add the data scaling results for the equivariant model for a 
    # selected lmax and width into the main results dictionary
    results["data"][lmax] = deepcopy(results_e3[lmax])
    for data_key in results["data"][lmax]:
        results["data"][lmax][data_key] = results["data"][lmax][data_key]["256"]
        
    # add the parameter scaling results for the equivariant model for a 
    # selected lmax and dataset size into the main results dictionary
    results["parameter"][lmax] = {}
    for param_key in ["16", "32", "64", "128", "256"]:
        results["parameter"][lmax][param_key] = results_e3[lmax]["20000"][param_key]

# Validation loss 1D NSLs

In [ ]:
# general setup
metric_key = "val_loss"
fit_functions = {
    "simple": r"Power Law",
    "floor": r"Power Law + Floor",
    "broken": r"Broken Power Law",
    "broken_with_amp": r"Broken power law + Amplitude",
}
model_names = [
    r"\textsc{OptiMetal2B} (CGC)", 
    r"\textsc{OptiMetal2B} (TC)", 
    r"\textsc{OptiMetal3B} (TC)",
    r"\textsc{UMA} ($\ell_\mathrm{max}=0$)", 
    r"\textsc{UMA} ($\ell_\mathrm{max}=1$)", 
    r"\textsc{UMA} ($\ell_\mathrm{max}=2$)",
    r"\textsc{UMA} ($\ell_\mathrm{max}=3$)",
]

"""
Try different fit functions for the data scaling.
"""

# try different fit functions for the data scaling and compare their goodness of fit
var = "D"
data_results = (
    results["data"]["2b"]["variant1"],
    results["data"]["2b"]["variant2"],
    results["data"]["3b"],
    results["data"]["lmax0"],
    results["data"]["lmax1"],
    results["data"]["lmax2"],
    results["data"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
for func_type, row_label in fit_functions.items():
    aicc_row = {m: np.nan for m in model_names}
    func = get_power_law(func_type)
    for name, data in zip(model_names, data_results):
        fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key)
        aicc_row[name] = np.round((fit_dict["aicc"]), 2)
    df.loc[row_label, model_names] = [aicc_row[m] for m in model_names]

# print and save the AICc table for the data scaling fits
latex_aicc = df.T.to_latex(
    index=True,
    escape=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    float_format=lambda x: f"{int(round(x))}" if np.isclose(x, round(x)) else f"{x:.2f}",
    column_format="l" + len(df.columns) * r"c@{\hspace{1em}}",
)
print("\nAICc table:")
print(latex_aicc)
with open(os.path.join(table_dir, f"{metric_key:s}_aicc_table_data.txt"), "w") as f:
    f.write(latex_aicc)
    
"""
BNSL data scaling fits.
"""

# fit all models with the BNSLs
var = "D"
data_results = (
    results["data"]["2b"]["variant1"],
    results["data"]["2b"]["variant2"],
    results["data"]["3b"],
    results["data"]["lmax0"],
    results["data"]["lmax1"],
    results["data"]["lmax2"],
    results["data"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
func_type = "broken"
func = get_power_law(func_type)
fit_params = []
for name, data in zip(model_names, data_results):
    fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key)
    fit_params.append([name, *list(fit_dict.values())])
# rows for the latex table
rows = build_rows_from_fits(fit_params, func_type=func_type, X=var)

# print and save the latex table
latex = latex_table_from_rows(rows, X="D", fit_func_display=False)
print(f"\nData scaling: {metric_key:s} & {func_type:s}")
print(latex)
with open(os.path.join(table_dir, f"{metric_key:s}_{func_type:s}_data_scaling_fit_parameter.txt"), "w") as f:
    f.write(latex)

"""
Saturating power law parameter scaling fits.
"""

# fit all models with the saturating power law
var = "N"
parameter_results = (
    results["parameter"]["2b"]["variant1"],
    results["parameter"]["2b"]["variant2"],
    results["parameter"]["3b"],
    results["parameter"]["lmax0"],
    results["parameter"]["lmax1"],
    results["parameter"]["lmax2"],
    results["parameter"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
func_type = "floor"
func = get_power_law(func_type)
fit_params = []
for name, data in zip(model_names, parameter_results):
    fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key, x_from_entry="num_parameter")
    fit_params.append([name, *list(fit_dict.values())])
# rows for the latex table
rows = build_rows_from_fits(fit_params, func_type=func_type, X=var)

# print and save the latex table
latex = latex_table_from_rows(rows, X="N", fit_func_display=False)
print(f"\nParameter scaling: {metric_key:s} & {func_type:s}")
print(latex)
with open(os.path.join(table_dir, f"{metric_key:s}_{func_type:s}_parameter_scaling_fit_parameter.txt"), "w") as f:
    f.write(latex)

In [ ]:
"""
Fig. 1
"""

# metric key
metric_key = "val_loss"

# plot with or without errorbars
error_bars = True

# figure setup
fig, axes = plt.subplots(2, 1, figsize=(3.5, 5))
fname = f"{metric_key:s}_scaling_laws"
    
# data scaling
var = "D"
data_func_type = "broken"
func = get_power_law(data_func_type)
    
# plot setup
ax = axes[0]
num_data = sorted([int(x) for x in results["data"]["2b"]["variant1"].keys()]) # tick positions for the x-axis
x_fit = np.logspace(np.log10(2000), np.log10(200000), 100)

# OptiMetal2B (CGC)
color = "k"
name = r"\textsc{OptiMetal2B} (CGC)"
data = results["data"]["2b"]["variant1"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)
        
# add "unbroken" power-law fit for comparison
data = _filter_nonempty(data)
if len(data) < 2:
    raise ValueError("Need at least two points for a fit")
sort_idx = np.argsort([int(x) for x in data.keys()])
x = np.array(sorted([int(x) for x in data.keys()]), dtype=float)
y_mean = np.array(
    [np.mean([entry[metric_key] for entry in data[k]]) for k in np.array(list(data.keys()))[sort_idx]], 
    dtype=float,
)
y_std = np.array(
    [np.std([entry[metric_key] for entry in data[k]]) for k in np.array(list(data.keys()))[sort_idx]], 
    dtype=float,
)
alpha_1, d0_1 = np.polyfit(np.log10(x[:3]), np.log10(y_mean[:3]), deg=1)
alpha_2, d0_2 = np.polyfit(np.log10(x[-3:]), np.log10(y_mean[-3:]), deg=1)
y_fit_unbroken_1 = 10**d0_1 * x_fit**alpha_1
ax.plot(x_fit, y_fit_unbroken_1, ":", color="k", zorder=-1)
y_fit_unbroken_2 = 10**d0_2 * x_fit**alpha_2
ax.plot(x_fit, y_fit_unbroken_2, ":", color="k", zorder=-1, label="Asymptotic fit")

# OptiMetal2B (TC)
color = "tab:orange"
name = r"\textsc{OptiMetal2B} (TC)"
data = results["data"]["2b"]["variant2"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal3B (TC)
color = "tab:green"
name = r"\textsc{OptiMetal3B} (TC)"
data = results["data"]["3b"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# UMA (lmax=2)
color = "tab:blue"
name = r"\textsc{UMA} ($\ell_\mathrm{max}=2$)"
data = results["data"]["lmax2"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(2000, 200000)
ax.set_ylim(bottom=0.4, top=2.3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_data)
ax.xaxis.set_major_locator(FixedLocator(num_data))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.tick_params(axis="x", which="minor", length=0)
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$D$")
ax.set_ylabel(r"$L_\mathrm{val}$")
leg_models = ax.legend(loc="lower left", handletextpad=0.25, handlelength=1.25)
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(data_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$N \approx 10\,\mathrm{M}$ ($d_\mathrm{h}=256$)", loc="upper right", handletextpad=0.25, handlelength=1.25)
ax.add_artist(leg_models)

# parameters scaling
var = "N"
param_func_type = "floor"
func = get_power_law(param_func_type)

# plot setup
ax = axes[1]
num_parameter = [1e5, 5e5, 1e6, 5e6, 1e7, 5e7] # tick positions for the x-axis
x_fit = np.logspace(np.log10(1e5), np.log10(2e8), 100)

# OptiMetal2B (CGC)
color = "k"
name = r"\textsc{OptiMetal2B} (CGC)"
data = results["parameter"]["2b"]["variant1"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1, label=get_fit_func_str(param_func_type, var))

# OptiMetal2B (TC)
color = "tab:orange"
name = r"\textsc{OptiMetal2B} (TC)"
data = results["parameter"]["2b"]["variant2"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal3B (TC)
color = "tab:green"
name = r"\textsc{OptiMetal3B} (TC)"
data = results["parameter"]["3b"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# UMA (lmax=2)
color = "tab:blue"
name = r"\textsc{UMA} ($\ell_\mathrm{max}=2$)"
data = results["parameter"]["lmax2"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(1e5, 2e8)
ax.set_ylim(top=1.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_parameter)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$N$")
ax.set_ylabel(r"$L_\mathrm{val}$")
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(param_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$D=20000$", loc="upper right", handletextpad=0.25, handlelength=1.25)

# save the figure
fig.tight_layout()
fig.align_labels()
fig.savefig(os.path.join(fig_dir, fname + ".pdf"))

In [ ]:
"""
Fig. 2
"""

# metric key
metric_key = "val_loss"

# plot with or without errorbars
error_bars = True

# figure setup
fig, axes = plt.subplots(2, 1, figsize=(3.5, 5))
fname = f"{metric_key:s}_uma_scaling_laws"
    
# data scaling
var = "D"
data_func_type = "broken"
func = get_power_law(data_func_type)
    
# plot setup
ax = axes[0]
num_data = sorted([int(x) for x in results["data"]["2b"]["variant1"].keys()]) # tick positions for the x-axis
x_fit = np.logspace(np.log10(2000), np.log10(200000), 100)

# UMA
colors = ["k", "tab:orange", "tab:blue", "tab:green"]
for lmax in range(4):
    name = rf"$\ell_\mathrm{{max}}={lmax:d}$"
    data = results["data"][f"lmax{lmax:d}"]
    plot_data(ax, data, name, key=metric_key, color=colors[lmax], error_bars=error_bars)
    fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
    y_fit = func(x_fit, *fit_dict["popt"])
    ax.plot(x_fit, y_fit, "--", color=colors[lmax], zorder=-1)

# log-log plot and axis ticks
ax.set_xlim(2000, 200000)
ax.set_ylim(bottom=0.4, top=2.3)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_data)
ax.xaxis.set_major_locator(FixedLocator(num_data))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.tick_params(axis="x", which="minor", length=0)
ax.yaxis.set_minor_locator(FixedLocator([0.5, 1.0, 1.5, 2.0]))
ax.yaxis.set_major_locator(FixedLocator([0.5, 1.0, 1.5, 2.0]))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$D$")
ax.set_ylabel(r"$L_\mathrm{val}$")
leg_models = ax.legend(title=r"\textsc{UMA}", loc="lower left", handletextpad=0.25, handlelength=1.25)
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(data_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$d_\mathrm{h}=256$", loc="upper right", handletextpad=0.25, handlelength=1.25)
ax.add_artist(leg_models)

# parameter scaling
var = "N"
param_func_type = "floor"
func = get_power_law(param_func_type)

# plot setup
ax = axes[1]
num_parameter = [5e4, 1e5, 5e5, 1e6, 5e6, 1e7, 5e7] # tick positions for the x-axis
x_fit = np.logspace(np.log10(5e4), np.log10(2e8), 100)

# UMA
colors = ["k", "tab:orange", "tab:blue", "tab:green"]
for lmax in range(4):
    data = results["parameter"][f"lmax{lmax:d}"]
    plot_data(ax, data, None, key=metric_key, color=colors[lmax], error_bars=error_bars, x_from_entry="num_parameter")
    fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
    y_fit = func(x_fit, *fit_dict["popt"])
    ax.plot(x_fit, y_fit, "--", color=colors[lmax], zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(5e4, 6e7)
ax.set_ylim(bottom=0.875, top=1.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_parameter)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$N$")
ax.set_ylabel(r"$L_\mathrm{val}$")
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(param_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$D=20000$", loc="upper right", handletextpad=0.25, handlelength=1.25)

# save the figure
fig.tight_layout()
fig.align_labels()
fig.savefig(os.path.join(fig_dir, fname + ".pdf"))

# Interband loss 1D NSLs

In [ ]:
# general setup
metric_key = "eps_loss"
fit_functions = {
    "simple": r"Power Law",
    "floor": r"Power Law + Floor",
    "broken": r"Broken Power Law",
    "broken_with_amp": r"Broken power law + Amplitude",
}
model_names = [
    r"\textsc{OptiMetal2B} (CGC)", 
    r"\textsc{OptiMetal2B} (TC)", 
    r"\textsc{OptiMetal3B} (TC)",
    r"\textsc{UMA} ($\ell_\mathrm{max}=0$)", 
    r"\textsc{UMA} ($\ell_\mathrm{max}=1$)", 
    r"\textsc{UMA} ($\ell_\mathrm{max}=2$)",
    r"\textsc{UMA} ($\ell_\mathrm{max}=3$)",
]

"""
Try different fit functions for the data scaling.
"""

# try different fit functions for the data scaling and compare their goodness of fit
var = "D"
data_results = (
    results["data"]["2b"]["variant1"],
    results["data"]["2b"]["variant2"],
    results["data"]["3b"],
    results["data"]["lmax0"],
    results["data"]["lmax1"],
    results["data"]["lmax2"],
    results["data"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
for func_type, row_label in fit_functions.items():
    aicc_row = {m: np.nan for m in model_names}
    func = get_power_law(func_type)
    for name, data in zip(model_names, data_results):
        fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key)
        aicc_row[name] = np.round((fit_dict["aicc"]), 2)
    df.loc[row_label, model_names] = [aicc_row[m] for m in model_names]

# print and save the AICc table for the data scaling fits
latex_aicc = df.T.to_latex(
    index=True,
    escape=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    float_format=lambda x: f"{int(round(x))}" if np.isclose(x, round(x)) else f"{x:.2f}",
    column_format="l" + len(df.columns) * r"c@{\hspace{1em}}",
)
print("\nAICc table:")
print(latex_aicc)
with open(os.path.join(table_dir, f"{metric_key:s}_aicc_table_data.txt"), "w") as f:
    f.write(latex_aicc)
    
"""
BNSL data scaling fits.
"""

# fit all models with the BNSLs
var = "D"
data_results = (
    results["data"]["2b"]["variant1"],
    results["data"]["2b"]["variant2"],
    results["data"]["3b"],
    results["data"]["lmax0"],
    results["data"]["lmax1"],
    results["data"]["lmax2"],
    results["data"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
func_type = "broken_with_amp"
func = get_power_law(func_type)
fit_params = []
for name, data in zip(model_names, data_results):
    fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key)
    fit_params.append([name, *list(fit_dict.values())])
# rows for the latex table
rows = build_rows_from_fits(fit_params, func_type=func_type, X=var)

# print and save the latex table
latex = latex_table_from_rows(rows, X="D", fit_func_display=False)
print(f"\nData scaling: {metric_key:s} & {func_type:s}")
print(latex)
with open(os.path.join(table_dir, f"{metric_key:s}_{func_type:s}_data_scaling_fit_parameter.txt"), "w") as f:
    f.write(latex)

"""
Saturating power law parameter scaling fits.
"""

# fit all models with the saturating power law
var = "N"
parameter_results = (
    results["parameter"]["2b"]["variant1"],
    results["parameter"]["2b"]["variant2"],
    results["parameter"]["3b"],
    results["parameter"]["lmax0"],
    results["parameter"]["lmax1"],
    results["parameter"]["lmax2"],
    results["parameter"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
func_type = "floor"
func = get_power_law(func_type)
fit_params = []
for name, data in zip(model_names, parameter_results):
    fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key, x_from_entry="num_parameter")
    fit_params.append([name, *list(fit_dict.values())])
# rows for the latex table
rows = build_rows_from_fits(fit_params, func_type=func_type, X=var)

# print and save the latex table
latex = latex_table_from_rows(rows, X="N", fit_func_display=False)
print(f"\nParameter scaling: {metric_key:s} & {func_type:s}")
print(latex)
with open(os.path.join(table_dir, f"{metric_key:s}_{func_type:s}_parameter_scaling_fit_parameter.txt"), "w") as f:
    f.write(latex)

In [ ]:
"""
SI Fig.
"""

# metric key
metric_key = "eps_loss"

# plot with or without errorbars
error_bars = True

# figure setup
fig, axes = plt.subplots(2, 1, figsize=(3.5, 5))
fname = f"{metric_key:s}_scaling_laws"
    
# data scaling
var = "D"
data_func_type = "broken_with_amp"
func = get_power_law(data_func_type)
    
# plot setup
ax = axes[0]
num_data = sorted([int(x) for x in results["data"]["2b"]["variant1"].keys()]) # tick positions for the x-axis
x_fit = np.logspace(np.log10(2000), np.log10(200000), 100)

# OptiMetal2B CGC
color = "k"
name = r"\textsc{OptiMetal2B} (CGC)"
data = results["data"]["2b"]["variant1"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal2B CGC
color = "tab:orange"
name = r"\textsc{OptiMetal2B} (TC)"
data = results["data"]["2b"]["variant2"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal3B
color = "tab:green"
name = r"\textsc{OptiMetal3B} (TC)"
data = results["data"]["3b"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# UMA (lmax=2)
color = "tab:blue"
name = r"\textsc{UMA} ($\ell_\mathrm{max}=2$)"
data = results["data"]["lmax2"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(2000, 200000)
ax.set_ylim(top=1.25)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_data)
ax.xaxis.set_major_locator(FixedLocator(num_data))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.tick_params(axis="x", which="minor", length=0)
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$D$")
ax.set_ylabel(r"$L_\mathrm{inter}$")
leg_models = ax.legend(loc="lower left", handletextpad=0.25, handlelength=1.25)
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(data_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$N \approx 10\,\mathrm{M}$ ($d_\mathrm{h}=256$)", loc="upper right", handletextpad=0.25, handlelength=1.25)
ax.add_artist(leg_models)

# parameters scaling
var = "N"
param_func_type = "floor"
func = get_power_law(param_func_type)

# plot setup
ax = axes[1]
num_parameter = [1e5, 5e5, 1e6, 5e6, 1e7, 5e7, 1e8] # tick positions for the x-axis
x_fit = np.logspace(np.log10(1e5), np.log10(2e8), 100)

# OptiMetal2B CGC
color = "k"
name = r"\textsc{OptiMetal2B} (CGC)"
data = results["parameter"]["2b"]["variant1"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1, label=get_fit_func_str(param_func_type, var))

# OptiMetal2B CGC
color = "tab:orange"
name = r"\textsc{OptiMetal2B} (TC)"
data = results["parameter"]["2b"]["variant2"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal3B
color = "tab:green"
name = r"\textsc{OptiMetal3B} (TC)"
data = results["parameter"]["3b"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# UMA (lmax=2)
color = "tab:blue"
name = r"\textsc{UMA} ($\ell_\mathrm{max}=2$)"
data = results["parameter"]["lmax2"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(1e5, 2e8)
ax.set_ylim(top=0.9)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_parameter)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$N$")
ax.set_ylabel(r"$L_\mathrm{inter}$")
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(param_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$D=20000$", loc="upper right", handletextpad=0.25, handlelength=1.25)

# save the figure
fig.tight_layout()
fig.align_labels()
fig.savefig(os.path.join(fig_dir, fname + ".pdf"))

In [ ]:
"""
SI Fig.
"""

# metric key
metric_key = "eps_loss"

# plot with or without errorbars
error_bars = True

# figure setup
fig, axes = plt.subplots(2, 1, figsize=(3.5, 5))
fname = f"{metric_key:s}_uma_scaling_laws"
    
# data scaling
var = "D"
data_func_type = "broken_with_amp"
func = get_power_law(data_func_type)
    
# plot setup
ax = axes[0]
num_data = sorted([int(x) for x in results["data"]["2b"]["variant1"].keys()]) # tick positions for the x-axis
x_fit = np.logspace(np.log10(2000), np.log10(200000), 100)

# UMA
colors = ["k", "tab:orange", "tab:blue", "tab:green"]
for lmax in range(4):
    name = rf"$\ell_\mathrm{{max}}={lmax:d}$"
    data = results["data"][f"lmax{lmax:d}"]
    plot_data(ax, data, name, key=metric_key, color=colors[lmax], error_bars=error_bars)
    fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
    y_fit = func(x_fit, *fit_dict["popt"])
    ax.plot(x_fit, y_fit, "--", color=colors[lmax], zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(2000, 200000)
ax.set_ylim(top=1.4)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_data)
ax.xaxis.set_major_locator(FixedLocator(num_data))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.tick_params(axis="x", which="minor", length=0)
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$D$")
ax.set_ylabel(r"$L_\mathrm{inter}$")
leg_models = ax.legend(title=r"\textsc{UMA}", loc="lower left", handletextpad=0.25, handlelength=1.25)
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(data_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$d_\mathrm{h}=256$", loc="upper right", handletextpad=0.25, handlelength=1.25)
ax.add_artist(leg_models)

# parameter scaling
var = "N"
param_func_type = "floor"
func = get_power_law(param_func_type)

# plot setup
ax = axes[1]
num_parameter = [5e4, 1e5, 5e5, 1e6, 5e6, 1e7, 5e7] # tick positions for the x-axis
x_fit = np.logspace(np.log10(5e4), np.log10(2e8), 100)

# UMA
colors = ["k", "tab:orange", "tab:blue", "tab:green"]
for lmax in range(4):
    data = results["parameter"][f"lmax{lmax:d}"]
    plot_data(ax, data, None, key=metric_key, color=colors[lmax], error_bars=error_bars, x_from_entry="num_parameter")
    fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
    y_fit = func(x_fit, *fit_dict["popt"])
    ax.plot(x_fit, y_fit, "--", color=colors[lmax], zorder=-1)

# log-log plot and axis ticks
yticks = ax.get_yticks()
ax.set_xlim(5e4, 6e7)
ax.set_ylim(top=0.95)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_parameter)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$N$")
ax.set_ylabel(r"$L_\mathrm{inter}$")
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(param_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$D=20000$", loc="upper right", handletextpad=0.25, handlelength=1.25)

# save the figure
fig.tight_layout()
fig.align_labels()
fig.savefig(os.path.join(fig_dir, fname + ".pdf"))

# Drude loss 1D NSLs

In [ ]:
# general setup
metric_key = "drude_loss"
fit_functions = {
    "simple": r"Power Law",
    "floor": r"Power Law + Floor",
    "broken": r"Broken Power Law",
    "broken_with_amp": r"Broken power law + Amplitude",
}
model_names = [
    r"\textsc{OptiMetal2B} (CGC)", 
    r"\textsc{OptiMetal2B} (TC)", 
    r"\textsc{OptiMetal3B} (TC)",
    r"\textsc{UMA} ($\ell_\mathrm{max}=0$)", 
    r"\textsc{UMA} ($\ell_\mathrm{max}=1$)", 
    r"\textsc{UMA} ($\ell_\mathrm{max}=2$)",
    r"\textsc{UMA} ($\ell_\mathrm{max}=3$)",
]

"""
Try different fit functions for the data scaling.
"""

# try different fit functions for the data scaling and compare their goodness of fit
var = "D"
data_results = (
    results["data"]["2b"]["variant1"],
    results["data"]["2b"]["variant2"],
    results["data"]["3b"],
    results["data"]["lmax0"],
    results["data"]["lmax1"],
    results["data"]["lmax2"],
    results["data"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
for func_type, row_label in fit_functions.items():
    aicc_row = {m: np.nan for m in model_names}
    func = get_power_law(func_type)
    for name, data in zip(model_names, data_results):
        fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key)
        aicc_row[name] = np.round((fit_dict["aicc"]), 2)
    df.loc[row_label, model_names] = [aicc_row[m] for m in model_names]

# print and save the AICc table for the data scaling fits
latex_aicc = df.T.to_latex(
    index=True,
    escape=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    float_format=lambda x: f"{int(round(x))}" if np.isclose(x, round(x)) else f"{x:.2f}",
    column_format="l" + len(df.columns) * r"c@{\hspace{1em}}",
)
print("\nAICc table:")
print(latex_aicc)
with open(os.path.join(table_dir, f"{metric_key:s}_aicc_table_data.txt"), "w") as f:
    f.write(latex_aicc)
    
"""
BNSL data scaling fits.
"""

# fit all models with the BNSLs
var = "D"
data_results = (
    results["data"]["2b"]["variant1"],
    results["data"]["2b"]["variant2"],
    results["data"]["3b"],
    results["data"]["lmax0"],
    results["data"]["lmax1"],
    results["data"]["lmax2"],
    results["data"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
func_type = "broken_with_amp"
func = get_power_law(func_type)
fit_params = []
for name, data in zip(model_names, data_results):
    fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key)
    fit_params.append([name, *list(fit_dict.values())])
# rows for the latex table
rows = build_rows_from_fits(fit_params, func_type=func_type, X=var)

# print and save the latex table
latex = latex_table_from_rows(rows, X="D", fit_func_display=False)
print(f"\nData scaling: {metric_key:s} & {func_type:s}")
print(latex)
with open(os.path.join(table_dir, f"{metric_key:s}_{func_type:s}_data_scaling_fit_parameter.txt"), "w") as f:
    f.write(latex)

"""
Saturating power law parameter scaling fits.
"""

# fit all models with the saturating power law
var = "N"
parameter_results = (
    results["parameter"]["2b"]["variant1"],
    results["parameter"]["2b"]["variant2"],
    results["parameter"]["3b"],
    results["parameter"]["lmax0"],
    results["parameter"]["lmax1"],
    results["parameter"]["lmax2"],
    results["parameter"]["lmax3"],
)
df = pd.DataFrame(
    index=list(fit_functions.values()),
    columns=model_names, 
    dtype=float,
)
func_type = "floor"
func = get_power_law(func_type)
fit_params = []
for name, data in zip(model_names, parameter_results):
    fit_dict = fit_scaling_law(data, func_type=func_type, key=metric_key, x_from_entry="num_parameter")
    fit_params.append([name, *list(fit_dict.values())])
# rows for the latex table
rows = build_rows_from_fits(fit_params, func_type=func_type, X=var)

# print and save the latex table
latex = latex_table_from_rows(rows, X="N", fit_func_display=False)
print(f"\nParameter scaling: {metric_key:s} & {func_type:s}")
print(latex)
with open(os.path.join(table_dir, f"{metric_key:s}_{func_type:s}_parameter_scaling_fit_parameter.txt"), "w") as f:
    f.write(latex)

In [ ]:
"""
SI Fig.
"""

# metric key
metric_key = "drude_loss"

# plot with or without errorbars
error_bars = True

# figure setup
fig, axes = plt.subplots(2, 1, figsize=(3.5, 5))
fname = f"{metric_key:s}_scaling_laws"
    
# data scaling
var = "D"
data_func_type = "broken_with_amp"
func = get_power_law(data_func_type)
    
# plot setup
ax = axes[0]
num_data = sorted([int(x) for x in results["data"]["2b"]["variant1"].keys()]) # tick positions for the x-axis
x_fit = np.logspace(np.log10(2000), np.log10(200000), 100)

# OptiMetal2B (CGC)
color = "k"
name = r"\textsc{OptiMetal2B} (CGC)"
data = results["data"]["2b"]["variant1"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal2B (TC)
color = "tab:orange"
name = r"\textsc{OptiMetal2B} (TC)"
data = results["data"]["2b"]["variant2"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal3B (TC)
color = "tab:green"
name = r"\textsc{OptiMetal3B} (TC)"
data = results["data"]["3b"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# UMA (lmax=2)
color = "tab:blue"
name = r"\textsc{UMA} ($\ell_\mathrm{max}=2$)"
data = results["data"]["lmax2"]
plot_data(ax, data, name, key=metric_key, color=color, error_bars=error_bars)
fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# log-log plot and axis ticks
yticks = [0.2, 0.4, 0.6]
ax.set_xlim(2000, 200000)
ax.set_ylim(bottom=0.11, top=0.8)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_data)
ax.xaxis.set_major_locator(FixedLocator(num_data))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.tick_params(axis="x", which="minor", length=0)
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$D$")
ax.set_ylabel(r"$L_\mathrm{D}$")
leg_models = ax.legend(loc="lower left", handletextpad=0.25, handlelength=1.25)
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(data_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$N \approx 10\,\mathrm{M}$ ($d_\mathrm{h}=256$)", loc="upper right", handletextpad=0.25, handlelength=1.25)
ax.add_artist(leg_models)

# parameters scaling
var = "N"
param_func_type = "floor"
func = get_power_law(param_func_type)

# plot setup
ax = axes[1]
num_parameter = [1e5, 5e5, 1e6, 5e6, 1e7, 5e7, 1e8] # tick positions for the x-axis
x_fit = np.logspace(np.log10(1e5), np.log10(2e8), 100)

# OptiMetal2B (CGC)
color = "k"
name = r"\textsc{OptiMetal2B} (CGC)"
data = results["parameter"]["2b"]["variant1"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1, label=get_fit_func_str(param_func_type, var))

# OptiMetal2B (TC)
color = "tab:orange"
name = r"\textsc{OptiMetal2B} (TC)"
data = results["parameter"]["2b"]["variant2"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# OptiMetal3B (TC)
color = "tab:green"
name = r"\textsc{OptiMetal3B} (TC)"
data = results["parameter"]["3b"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# UMA (lmax=2)
color = "tab:blue"
name = r"\textsc{UMA} ($\ell_\mathrm{max}=2$)"
data = results["parameter"]["lmax2"]
plot_data(ax, data, None, key=metric_key, color=color, error_bars=error_bars, x_from_entry="num_parameter")
fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
y_fit = func(x_fit, *fit_dict["popt"])
ax.plot(x_fit, y_fit, "--", color=color, zorder=-1)

# log-log plot and axis ticks
yticks = [0.3, 0.4, 0.5]
ax.set_xlim(1e5, 2e8)
ax.set_ylim(bottom=0.29, top=0.5)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_parameter)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$N$")
ax.set_ylabel(r"$L_\mathrm{D}$")
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(param_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$D=20000$", loc="upper right", handletextpad=0.25, handlelength=1.25)

# save the figure
fig.tight_layout()
fig.align_labels()
fig.savefig(os.path.join(fig_dir, fname + ".pdf"))

In [ ]:
"""
SI Fig.
"""

# metric key
metric_key = "drude_loss"

# plot with or without errorbars
error_bars = True

# figure setup
fig, axes = plt.subplots(2, 1, figsize=(3.5, 5))
fname = f"{metric_key:s}_uma_scaling_laws"
    
# data scaling
var = "D"
data_func_type = "broken_with_amp"
func = get_power_law(data_func_type)
    
# plot setup
ax = axes[0]
num_data = sorted([int(x) for x in results["data"]["2b"]["variant1"].keys()]) # tick positions for the x-axis
x_fit = np.logspace(np.log10(2000), np.log10(200000), 100)

# UMA
colors = ["k", "tab:orange", "tab:blue", "tab:green"]
for lmax in range(4):
    name = rf"$\ell_\mathrm{{max}}={lmax:d}$"
    data = results["data"][f"lmax{lmax:d}"]
    plot_data(ax, data, name, key=metric_key, color=colors[lmax], error_bars=error_bars)
    fit_dict = fit_scaling_law(data, func_type=data_func_type, key=metric_key)
    y_fit = func(x_fit, *fit_dict["popt"])
    ax.plot(x_fit, y_fit, "--", color=colors[lmax], zorder=-1)

# log-log plot and axis ticks
yticks =[0.2, 0.4, 0.6, 0.8]
ax.set_xlim(2000, 200000)
ax.set_ylim(top=1.0)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_data)
ax.xaxis.set_major_locator(FixedLocator(num_data))
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.tick_params(axis="x", which="minor", length=0)
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$D$")
ax.set_ylabel(r"$L_\mathrm{D}$")
leg_models = ax.legend(title=r"\textsc{UMA}", loc="lower left", handletextpad=0.25, handlelength=1.25)
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(data_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$d_\mathrm{h}=256$", loc="upper right", handletextpad=0.25, handlelength=1.25)
ax.add_artist(leg_models)

# parameter scaling
var = "N"
param_func_type = "floor"
func = get_power_law(param_func_type)

# plot setup
ax = axes[1]
num_parameter = [5e4, 1e5, 5e5, 1e6, 5e6, 1e7, 5e7] # tick positions for the x-axis
x_fit = np.logspace(np.log10(5e4), np.log10(2e8), 100)

# UMA
colors = ["k", "tab:orange", "tab:blue", "tab:green"]
for lmax in range(4):
    data = results["parameter"][f"lmax{lmax:d}"]
    plot_data(ax, data, None, key=metric_key, color=colors[lmax], error_bars=error_bars, x_from_entry="num_parameter")
    fit_dict = fit_scaling_law(data, func_type=param_func_type, key=metric_key, x_from_entry="num_parameter")
    y_fit = func(x_fit, *fit_dict["popt"])
    ax.plot(x_fit, y_fit, "--", color=colors[lmax], zorder=-1)

# log-log plot and axis ticks
yticks = [0.3, 0.4, 0.5]
ax.set_xlim(5e4, 6e7)
ax.set_ylim(bottom=0.29, top=0.55)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xticks(num_parameter)
ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
ax.yaxis.set_minor_locator(FixedLocator(yticks))
ax.yaxis.set_major_locator(FixedLocator(yticks))
ax.yaxis.set_minor_formatter(ScalarFormatter())
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.tick_params(axis="y", which="minor", length=0)

# axis labels and legends
ax.set_xlabel(r"$N$")
ax.set_ylabel(r"$L_\mathrm{D}$")
fit_handle = [Line2D([], [], ls="--", color="tab:gray")]
fit_label = [get_fit_func_str(param_func_type, var)]
leg = ax.legend(fit_handle, fit_label, title=r"$D=20000$", loc="upper right", handletextpad=0.25, handlelength=1.25)

# save the figure
fig.tight_layout()
fig.align_labels()
fig.savefig(os.path.join(fig_dir, fname + ".pdf"))

# Load 2D NSL data

In [ ]:
"""
Load the data for the invariant models.
"""

# path of the JSON file where results are stored
json_path = os.path.join("research", "scaling_data", "scaling_grid_results.json")

# load the results if the file exists, otherwise print a warning
if os.path.exists(json_path): 
    with open(json_path, "r") as f:
        results = json.load(f)
    print(f"Loaded results for {len(extract_processed_dirs(results)):d} models from JSON file")
else:
    raise FileNotFoundError(f"File not found: {json_path:s}.")
    
"""
Load the data for the equivariant model.
"""

# path of the JSON file where results are stored
json_path_e3 = os.path.join("research_e3", "scaling_data", "scaling_results.json")

# load the results if the file exists, otherwise print a warning
if os.path.exists(json_path_e3): 
    with open(json_path_e3, "r") as f: 
        results_e3 = json.load(f)
    print(f"Loaded results for {len(extract_processed_dirs(results_e3)):d} models from JSON file")
else:
    raise FileNotFoundError(f"File not found: {json_path_e3:s}.")

# add the results for the equivariant model to the results dictionary
for key in results_e3:
    results[key] = results_e3[key]

# 2D NSL maps

In [ ]:
# general setup
metric_key = "val_loss"
fit_functions = {
    "kaplan": (
        "Kaplan-style NSL",
        global_scaling_fit_kaplan, 
        [0.5, 0.5, 0.5, 1e4, 1e4],
    ),
    "hoffmann": (
        "Hoffmann-style NSL", 
        global_scaling_fit_hoffmann, 
        [0.5, 0.5, 0.5, 1e4, 1e4, 0],
    ),
}
model_names = {
    "2b": r"\textsc{OptiMetal2B} (TC)", 
    "3b": r"\textsc{OptiMetal3B} (TC)",
    "lmax0": r"\textsc{UMA} ($\ell_\mathrm{max}=0$)", 
    "lmax1": r"\textsc{UMA} ($\ell_\mathrm{max}=1$)", 
    "lmax2": r"\textsc{UMA} ($\ell_\mathrm{max}=2$)",
    "lmax3": r"\textsc{UMA} ($\ell_\mathrm{max}=3$)",
}

"""
Try different fit functions without the UMA width = 16 model.
"""

# try different fit functions for the data scaling and compare their goodness of fit
df = pd.DataFrame(
    index=[label for label, _, _ in fit_functions.values()],
    columns=list(model_names.values()),
    dtype=float,
)
fit_params = []
for name, (label, func, p0) in fit_functions.items():
    aicc_row = {m: np.nan for m in model_names}
    for model in model_names:
        data = deepcopy(results[model])
        if "lmax" in model:
            for d in data:
                for w in data[d]:
                    if w == "16":
                        del data[d]["16"]
                        break
        datapoints, widths, num_parameters, mus, stds = grid_to_vectors(data, metric_key=metric_key)
        x_grid = (datapoints.ravel(), num_parameters.ravel())
        popt, pcov = curve_fit(
            f=func,
            xdata=x_grid,
            ydata=mus.ravel(),
            p0=p0,
            sigma=stds.ravel(),
            absolute_sigma=True,
            maxfev=100000,
        )
        perr = np.sqrt(np.diag(pcov))
        y_fit = func(x_grid, *popt)
        res = mus.ravel() - y_fit
        aicc = calc_aicc(res, len(popt))
        aicc_row[model] = np.round(aicc, 2)
        if name == "kaplan":
            row = {
                "Model": model_names[model],
                r"$\alpha_N$": format_val(popt[0]),
                r"$\alpha_{D,1}$": format_val(popt[1]),
                r"$\alpha_{D,2}$": format_val(popt[2]),
                r"$N_c$": fmt_param(popt[3]),
                r"$D_c$": fmt_param(popt[4]),
            }
            fit_params.append(row)
    df.loc[label, list(model_names.values())] = [aicc_row[m] for m in model_names]
    
# print and save the AICc table for the data scaling fits
latex_aicc = df.T.to_latex(
    index=True,
    escape=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    float_format=lambda x: f"{int(round(x))}" if np.isclose(x, round(x)) else f"{x:.2f}",
    column_format="l" + len(df.columns) * r"c@{\hspace{1em}}",
)
print("\nAICc table:")
print(latex_aicc)
with open(os.path.join(table_dir, f"{metric_key:s}_aicc_table_2d_wo_uma_width_16.txt"), "w") as f:
    f.write(latex_aicc)
    
# print and save the fit parameter latex table
df = pd.DataFrame(fit_params)
latex_params_global = df.to_latex(
    index=False,
    escape=False,
    longtable=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    column_format=len(df.columns) * r"c@{\hspace{1em}}",
)
print(f"Fit parameters: kaplan & {metric_key:s}")
print(latex_params_global)
with open(os.path.join(table_dir, f"{metric_key:s}_kaplan_fit_parameter_wo_uma_width_16.txt"), "w") as f:
    f.write(latex_params_global)

In [ ]:
# general setup
metric_key = "val_loss"
fit_functions = {
    "kaplan": (
        "Kaplan-style NSL",
        global_scaling_fit_kaplan, 
        [0.5, 0.5, 0.5, 1e4, 1e4],
    ),
    "hoffmann": (
        "Hoffmann-style NSL", 
        global_scaling_fit_hoffmann, 
        [0.5, 0.5, 0.5, 1e4, 1e4, 0],
    ),
}
model_names = {
    "2b": r"\textsc{OptiMetal2B} (TC)", 
    "3b": r"\textsc{OptiMetal3B} (TC)",
    "lmax0": r"\textsc{UMA} ($\ell_\mathrm{max}=0$)", 
    "lmax1": r"\textsc{UMA} ($\ell_\mathrm{max}=1$)", 
    "lmax2": r"\textsc{UMA} ($\ell_\mathrm{max}=2$)",
    "lmax3": r"\textsc{UMA} ($\ell_\mathrm{max}=3$)",
}

"""
Try different fit functions.
"""

# try different fit functions for the data scaling and compare their goodness of fit
df = pd.DataFrame(
    index=[label for label, _, _ in fit_functions.values()],
    columns=list(model_names.values()),
    dtype=float,
)
fit_params = []
for name, (label, func, p0) in fit_functions.items():
    aicc_row = {m: np.nan for m in model_names}
    for model in model_names:
        datapoints, widths, num_parameters, mus, stds = grid_to_vectors(results[model], metric_key=metric_key)
        x_grid = (datapoints.ravel(), num_parameters.ravel())
        popt, pcov = curve_fit(
            f=func,
            xdata=x_grid,
            ydata=mus.ravel(),
            p0=p0,
            sigma=stds.ravel(),
            absolute_sigma=True,
            maxfev=100000,
        )
        perr = np.sqrt(np.diag(pcov))
        y_fit = func(x_grid, *popt)
        res = mus.ravel() - y_fit
        aicc = calc_aicc(res, len(popt))
        aicc_row[model] = np.round(aicc, 2)
        if name == "kaplan":
            row = {
                "Model": model_names[model],
                r"$\alpha_N$": format_val(popt[0]),
                r"$\alpha_{D,1}$": format_val(popt[1]),
                r"$\alpha_{D,2}$": format_val(popt[2]),
                r"$N_c$": fmt_param(popt[3]),
                r"$D_c$": fmt_param(popt[4]),
            }
            fit_params.append(row)
    df.loc[label, list(model_names.values())] = [aicc_row[m] for m in model_names]
    
# print and save the AICc table for the data scaling fits
latex_aicc = df.T.to_latex(
    index=True,
    escape=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    float_format=lambda x: f"{int(round(x))}" if np.isclose(x, round(x)) else f"{x:.2f}",
    column_format="l" + len(df.columns) * r"c@{\hspace{1em}}",
)
print("\nAICc table:")
print(latex_aicc)
with open(os.path.join(table_dir, f"{metric_key:s}_aicc_table_2d.txt"), "w") as f:
    f.write(latex_aicc)
    
# print and save the fit parameter latex table
df = pd.DataFrame(fit_params)
latex_params_global = df.to_latex(
    index=False,
    escape=False,
    longtable=False,
    multicolumn=True,
    multicolumn_format="c",
    bold_rows=False,
    column_format=len(df.columns) * r"c@{\hspace{1em}}",
)
print(f"Fit parameters: kaplan & {metric_key:s}")
print(latex_params_global)
with open(os.path.join(table_dir, f"{metric_key:s}_kaplan_fit_parameter.txt"), "w") as f:
    f.write(latex_params_global)

In [ ]:
"""
Fig. 3
"""

# general setup
metric_key = "val_loss"
model_names = {
    "2b": r"\textsc{OptiMetal2B} (TC)", 
    "3b": r"\textsc{OptiMetal3B} (TC)",
    "lmax0": r"\textsc{UMA} ($\ell_\mathrm{max}=0$)", 
    "lmax1": r"\textsc{UMA} ($\ell_\mathrm{max}=1$)", 
    "lmax2": r"\textsc{UMA} ($\ell_\mathrm{max}=2$)",
    "lmax3": r"\textsc{UMA} ($\ell_\mathrm{max}=3$)",
}

# fit setup
func = global_scaling_fit_kaplan
p0 = [0.5, 0.5, 0.5, 1e4, 1e4]

# plot setup
cmap_name = "viridis_r"
cmap = plt.get_cmap(cmap_name)
error_bars = True
yticks = [0.5, 1.0, 1.5, 2.0]

# loop over all models
for model in model_names:
    
    # figure setup
    fig = plt.figure(figsize=(3.5, 3.75), constrained_layout=True)
    gs = fig.add_gridspec(2, 2, width_ratios=[1.0, 0.1])

    # fit the function to the data
    datapoints, widths, num_parameters, mus, stds = grid_to_vectors(results[model], metric_key=metric_key)
    x_grid = (datapoints.ravel(), num_parameters.ravel())
    popt, _ = curve_fit(
        f=func,
        xdata=x_grid,
        ydata=mus.ravel(),
        p0=p0,
        sigma=stds.ravel(),
        absolute_sigma=True,
        maxfev=100000,
    )
    y_fit = func(x_grid, *popt)

    # plot the valdation loss over N for every D
    ax = fig.add_subplot(gs[0, 0])  
    norm_d = LogNorm(vmin=np.min(datapoints), vmax=np.max(datapoints))
    for i in range(num_parameters.shape[0]):
        if error_bars:
            ax.errorbar(
                num_parameters[i, :],
                mus[i, :],
                yerr=stds[i, :],
                fmt="o",
                markersize=4,
                markeredgecolor=cmap(norm_d(datapoints[i, 0])),
                markerfacecolor=cmap(norm_d(datapoints[i, 0])),
                ecolor=cmap(norm_d(datapoints[i, 0])),
                capsize=4,
                linestyle="none",
                label=fmt_param(datapoints[i, 0]),
            ) 
        else:
            ax.plot(
                num_parameters[i, :], 
                mus[i, :], 
                "o", 
                label=fmt_param(datapoints[i, 0]), 
                color=cmap(norm_d(datapoints[i, 0])),
            )
        nmin = np.nanmin(num_parameters[i, :])
        nmax = np.nanmax(num_parameters[i, :])
        if "lmax0" in model or "lmax1" in model:
            xfit = np.logspace(np.log10(5e4), np.log10(1e7), 200)
        elif "lmax2" in model or "lmax3" in model:
            xfit = np.logspace(np.log10(5e4), np.log10(5e7), 200)
        else:
            xfit = np.logspace(np.log10(2e5), np.log10(2e8), 200)
        yfit = func((np.full_like(xfit, datapoints[i, 0]), xfit), *popt)
        ax.plot(xfit, yfit, "--", color=cmap(norm_d(datapoints[i, 0])), zorder=-1)
    ax.set_xlabel(r"$N$")
    ax.set_ylabel(r"$L_\mathrm{val}$")
    ax.set_title(model_names[model], pad=3)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
    ax.set_xlim(xfit[0], xfit[-1])
    ax.yaxis.set_minor_locator(FixedLocator(yticks))
    ax.yaxis.set_major_locator(FixedLocator(yticks))
    ax.yaxis.set_minor_formatter(ScalarFormatter())
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.tick_params(axis="y", which="minor", length=0)
    if "lmax0" in model or "lmax1" in model:
        ax.set_ylim(0.5, 2.25)
    elif "lmax2" in model or "lmax3" in model:
        ax.set_ylim(0.4, 2.10)
    else:
        ax.set_ylim(0.4, 2.00)
    sm_d = plt.cm.ScalarMappable(norm=norm_d, cmap=cmap)
    sm_d.set_array([])
    cbar_top2 = fig.colorbar(sm_d, ax=ax, pad=0.02, label=r"$D$")
    cbar_top2.set_ticks(datapoints[:, 0])
    cbar_top2.ax.set_ylim(np.min(datapoints[:, 0]), np.max(datapoints[:, 0]))
    cbar_top2.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(x, remove_trailing_zeros=True)))
    
    # plot the valdation loss over D for every N
    ax = fig.add_subplot(gs[1, 0]) 
    norm_n = LogNorm(vmin=np.min(num_parameters), vmax=np.max(num_parameters))
    for i in range(datapoints.shape[1]):
        color = cmap(norm_n(num_parameters[0, i]))
        if error_bars:
            ax.errorbar(
                datapoints[:, i],
                mus[:, i],
                yerr=stds[:, i],
                fmt="o",
                markersize=4,
                markeredgecolor=color,
                markerfacecolor=color,
                ecolor=color,
                capsize=4,
                linestyle="none",
            ) 
        else:
            ax.plot(datapoints[:, i], mus[:, i], "o", ms=4, color=color)
        data_col = datapoints[:, i]
        xfit = np.logspace(np.log10(2000), np.log10(200000), 200)
        yfit = func((xfit, np.full_like(xfit, num_parameters[0, i])), *popt)
        ax.plot(xfit, yfit, "--", color=cmap(norm_n(num_parameters[0, i])), zorder=-1)
    ax.set_xlabel(r"$D$")
    ax.set_ylabel(r"$L_\mathrm{val}$")
    ax.set_xlim(2000, 200000)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xticks(np.unique(datapoints))
    ax.xaxis.set_major_locator(FixedLocator(np.unique(datapoints)))
    ax.xaxis.set_major_formatter(ScalarFormatter())
    ax.xaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(int(x), remove_trailing_zeros=True)))
    ax.yaxis.set_minor_locator(FixedLocator(yticks))
    ax.yaxis.set_major_locator(FixedLocator(yticks))
    ax.yaxis.set_minor_formatter(ScalarFormatter())
    ax.yaxis.set_major_formatter(ScalarFormatter())
    ax.tick_params(axis="y", which="minor", length=0)
    sm_np = plt.cm.ScalarMappable(norm=norm_n, cmap=cmap)
    sm_np.set_array([])
    cbar_top1 = fig.colorbar(sm_np, ax=ax, pad=0.02, label=r"$N$")
    cbar_top1.set_ticks(num_parameters[0, :])
    cbar_top1.ax.set_ylim(np.min(num_parameters[0, :]), np.max(num_parameters[0, :]))
    cbar_top1.ax.yaxis.set_major_formatter(FuncFormatter(lambda x, _: fmt_param(x, remove_trailing_zeros=False)))
    xlim = ax.get_xlim()
    if error_bars:
        ax.errorbar(
            [1, 2],
            [101, 102],
            yerr=[0.1, 0.2],
            fmt="o",
            markersize=4,
            markeredgecolor="k",
            markerfacecolor="k",
            ecolor="k",
            capsize=4,
            linestyle="none",
            label="Data",
        ) 
    else:
        ax.plot([1, 2], [101, 102], "o", ms=4, color="k", label="Data")
    ax.plot([1, 2], [101, 102], "--", color="k", label="Fit")
    ax.legend(handlelength=1.25, loc="lower left")
    ax.set_xlim(xlim)
    if "lmax0" in model or "lmax1" in model:
        ax.set_ylim(0.5, 2.25)
    elif "lmax2" in model or "lmax3" in model:
        ax.set_ylim(0.4, 2.10)
    else:
        ax.set_ylim(0.4, 2.00)
    handles, labels = ax.get_legend_handles_labels()
    order = [labels.index("Data"), labels.index("Fit")] # desired ordering
    ax.legend(
        [handles[i] for i in order], [labels[i] for i in order], 
        handlelength=1.25, 
        loc="lower left",
    )
    
    # align labels and save the figure
    fig.align_labels()
    fig.savefig(os.path.join(fig_dir, f"{metric_key:s}_{model:s}_scaling_map.svg"))